In [18]:
import pandas as pd
import random

# -----------------------------
# Base Sentences
# -----------------------------

human_sentences = [
    "i forgot to submit my assignment lol",
    "this code is kinda confusing tbh",
    "we should improve this logic",
    "the movie was really good",
    "idk why this is not working",
    "let’s fix this later",
    "this looks wrong to me",
    "i enjoyed working on this",

    # formal human (IMPORTANT)
    "The results indicate a need for improvement",
    "This approach may not be efficient",
    "The system requires further analysis",
    "We need to optimize the performance"
]

ai_sentences = [
    "The assignment was not submitted on time.",
    "This code exhibits a certain level of complexity.",
    "The logic should be improved for better efficiency.",
    "The movie provided an enjoyable experience.",
    "The issue persists despite multiple attempts.",
    "This can be resolved in a later stage.",
    "The result appears to be incorrect.",
    "The project was completed successfully.",

    # casual AI (IMPORTANT)
    "this kinda doesn’t make sense tbh",
    "maybe try something else here",
    "this feels a bit off",
    "not sure why this is happening"
]

# -----------------------------
# Noise Function
# -----------------------------

def add_noise(text):
    noise = [
        text,
        text.lower(),
        text.upper(),
        text + "!",
        text + "...",
        text.replace(" ", "  "),
        text.replace("is", "iz"),
        text.replace("to", "2"),
        text.replace("you", "u"),
    ]
    return random.choice(noise)

# -----------------------------
# Dataset Generation
# -----------------------------

rows = []
target = 50000

for _ in range(target // 2):

    # pick base
    h = random.choice(human_sentences)
    a = random.choice(ai_sentences)

    # add noise
    h = add_noise(h)
    a = add_noise(a)

    # -------------------------
    # HARD CASES (key fix)
    # -------------------------

    # AI looks human
    if random.random() < 0.3:
        a = "lol " + a.lower()

    # Human looks AI
    if random.random() < 0.3:
        h = "Therefore, " + h.capitalize()

    # Confusion mixing (VERY IMPORTANT)
    if random.random() < 0.2:
        a = random.choice(human_sentences)   # AI becomes human-like

    if random.random() < 0.2:
        h = random.choice(ai_sentences)      # Human becomes AI-like

    rows.append([h, 0])  # Human
    rows.append([a, 1])  # AI

# -----------------------------
# Create DataFrame
# -----------------------------

df = pd.DataFrame(rows, columns=["text", "label"])

# Shuffle dataset
df = df.sample(frac=1).reset_index(drop=True)

# Save file
df.to_csv("ai_detection_dataset_final.csv", index=False)

print("Dataset created:", df.shape)
print("\nLabel Distribution:\n", df["label"].value_counts())

Dataset created: (50000, 2)

Label Distribution:
 label
0    25000
1    25000
Name: count, dtype: int64


In [19]:
# SBERT-based AI Detection Model

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sentence_transformers import SentenceTransformer


# -------------------------------
# 1 Load Dataset
# -------------------------------

print("Loading dataset...\n")

df = pd.read_csv("ai_detection_dataset_final.csv")

print(df.head())
print("\nDataset size:", df.shape)
print("\nLabel Distribution:\n", df["label"].value_counts())

texts = df["text"]
labels = df["label"]


# -------------------------------
# 2 Train Test Split
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


# -------------------------------
# 3 Load SBERT Model
# -------------------------------

print("\nLoading SBERT model...\n")

sbert = SentenceTransformer("all-MiniLM-L6-v2")


# -------------------------------
# 4 Generate Embeddings
# -------------------------------

print("Generating embeddings...\n")

X_train_emb = sbert.encode(
    X_train.tolist(),
    show_progress_bar=True
)

X_test_emb = sbert.encode(
    X_test.tolist(),
    show_progress_bar=True
)


# -------------------------------
# 5 Train Classifier
# -------------------------------

print("\nTraining classifier...\n")

model = LogisticRegression(
    max_iter=300,
    class_weight='balanced'
)

model.fit(X_train_emb, y_train)


# -------------------------------
# 6 Predictions
# -------------------------------

predictions = model.predict(X_test_emb)

print("\nPrediction Distribution:")
print(np.unique(predictions, return_counts=True))


# -------------------------------
# 7 Evaluation
# -------------------------------

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print("\nModel Evaluation\n")

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, predictions))

print("\nClassification Report\n")
print(classification_report(y_test, predictions))


# -------------------------------
# 8 AI Detection Function
# -------------------------------

def detect_ai(text):

    emb = sbert.encode([text])
    pred = model.predict(emb)[0]

    return "AI Generated" if pred == 1 else "Human Written"


# -------------------------------
# 9 Example Test
# -------------------------------

print("\nExample Tests\n")

sample1 = "I forgot my charger again lol"
sample2 = "The absence of the charger resulted in inconvenience"

print(sample1, "→", detect_ai(sample1))
print(sample2, "→", detect_ai(sample2))


# -------------------------------
# 10 User Input Mode (CLI)
# -------------------------------

if __name__ == "__main__":

    print("\n--- AI Detection User Test ---")

    while True:

        user_text = input("\nEnter text: ")

        print("Prediction:", detect_ai(user_text))

        again = input("\nTest again? (y/n): ")

        if again.lower() != "y":
            print("\nProgram Ended.")
            break

Loading dataset...

                                text  label
0      the  movie  was  really  good      0
1  this kinda doesn’t make sense tbh      1
2        idk why thiz iz not working      0
3  this kinda doesn’t make sense tbh      0
4      maybe try something else here      1

Dataset size: (50000, 2)

Label Distribution:
 label
0    25000
1    25000
Name: count, dtype: int64

Training samples: 40000
Testing samples: 10000

Loading SBERT model...



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings...



Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]


Training classifier...


Prediction Distribution:
(array([0, 1]), array([4995, 5005]))

Model Evaluation

Accuracy : 0.8021
Precision: 0.7998001998001998
Recall   : 0.8038152610441767
F1 Score : 0.8018027040560841

Confusion Matrix

[[4018 1002]
 [ 977 4003]]

Classification Report

              precision    recall  f1-score   support

           0       0.80      0.80      0.80      5020
           1       0.80      0.80      0.80      4980

    accuracy                           0.80     10000
   macro avg       0.80      0.80      0.80     10000
weighted avg       0.80      0.80      0.80     10000


Example Tests

I forgot my charger again lol → AI Generated
The absence of the charger resulted in inconvenience → AI Generated

--- AI Detection User Test ---

Enter text: The evening involved ordering pizza and engaging in movie viewing as a leisure activity.
Prediction: Human Written

Test again? (y/n): y

Enter text: The evening involved ordering pizza and engaging in movie viewin